In [1]:
import pandas as pd
import numpy as np
import torch

In [2]:
from ultralytics import YOLO

In [3]:
m1 = YOLO(r"D:\Research\msc_research\yolo\Ultralytics\ultralytics-101\ultralytics\ultralytics\cfg\models\12\yolo12-bifpn.yaml",verbose=True)

WARNING no model scale passed. Assuming scale='n'.

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     98816  ultralytics.nn.modules.block.A2C2f           [128, 128, 1, True, 4]        
  7                  -1  1    147712

TypeError: list indices must be integers or slices, not list

In [8]:
from ultralytics.nn.modules import BiFPN

In [14]:
sample = torch.randn(64,224,224)

In [16]:
bi = BiFPN(channels=64)

In [12]:
bi

BiFPN(
  (layers): ModuleList(
    (0-2): 3 x BiFPNLayer(
      (wf_P6_td): WeightedFusion()
      (conv_P6_td): SeparableConv2d(
        (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (pointwise): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (wf_P5_td): WeightedFusion()
      (conv_P5_td): SeparableConv2d(
        (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
        (pointwise): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (wf_P4_td): WeightedFusion()
      (conv_P4_td): SeparableConv2d(
        (depthwise): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1

In [15]:
out = bi(sample)

ValueError: too many values to unpack (expected 5)

## BiFPN code with GEMINI
- `BiFPN`

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class WeightedFeatureFusion(nn.Module):
    def __init__(self, in_nodes, epsilon=1e-4):
        super(WeightedFeatureFusion, self).__init__()
        self.epsilon = epsilon
        # We need one weight per input node. 
        # Initialize with equal importance.
        self.weights = nn.Parameter(torch.ones(in_nodes, dtype=torch.float32), requires_grad=True)

    def forward(self, inputs):
        # inputs is a list of tensors [x1, x2, ...]
        assert len(inputs) == len(self.weights)
        
        # Apply ReLU to ensure non-negative weights
        w = F.relu(self.weights)
        
        # Normalize weights (Fast Normalized Fusion)
        w = w / (torch.sum(w, dim=0) + self.epsilon)
        
        # Weighted sum: w0*x0 + w1*x1 + ...
        # We expand w to allow broadcasting: w[0] -> (1,1,1,1)
        fusion = 0
        for i, x in enumerate(inputs):
            fusion += w[i] * x
            
        return fusion

In [3]:
class BiFPNBlock(nn.Module):
    def __init__(self, num_channels):
        super(BiFPNBlock, self).__init__()
        
        # Conv layers to process the fused features
        # Depthwise Separable Conv is used in the paper for efficiency
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(num_channels, num_channels, 3, 1, 1, groups=num_channels, bias=False),
                nn.Conv2d(num_channels, num_channels, 1, 1, 0, bias=True),
                nn.BatchNorm2d(num_channels),
                nn.SiLU(inplace=True) # Swish activation
            ) for _ in range(5 + 3) # 5 bottom-up + 3 intermediate top-down
        ])
        
        # Weighted Fusion layers
        # Top-down pathway fuses 2 inputs (Level i and Level i+1)
        self.td_fusions = nn.ModuleList([
            WeightedFeatureFusion(2) for _ in range(3) # P6, P5, P4 (P7 is the top, no fusion)
        ])
        
        # Bottom-up pathway fuses 3 inputs (Original i, Top-down i, Bottom-up i-1)
        # Except P3 and P7 which fuse 2 inputs
        self.bu_fusions = nn.ModuleList([
            WeightedFeatureFusion(2), # P3 (Original + Top-down)
            WeightedFeatureFusion(3), # P4
            WeightedFeatureFusion(3), # P5
            WeightedFeatureFusion(3), # P6
            WeightedFeatureFusion(2)  # P7 (Original + Bottom-up)
        ])
        
    def forward(self, features):
        # Features: [P3, P4, P5, P6, P7]
        p3_in, p4_in, p5_in, p6_in, p7_in = features
        
        # --- Top-Down Pathway ---
        # P7 stays as is for the top-down path
        p7_td = p7_in 
        
        # P6_td = Conv(Fusion(P6_in, Resize(P7_td)))
        p6_td = self.convs[0](
            self.td_fusions[0]([p6_in, F.interpolate(p7_td, scale_factor=2, mode='nearest')])
        )
        
        # P5_td = Conv(Fusion(P5_in, Resize(P6_td)))
        p5_td = self.convs[1](
            self.td_fusions[1]([p5_in, F.interpolate(p6_td, scale_factor=2, mode='nearest')])
        )
        
        # P4_td = Conv(Fusion(P4_in, Resize(P5_td)))
        p4_td = self.convs[2](
            self.td_fusions[2]([p4_in, F.interpolate(p5_td, scale_factor=2, mode='nearest')])
        )
        
        # --- Bottom-Up Pathway ---
        # P3_out = Conv(Fusion(P3_in, Resize(P4_td))) -> P3 is the bottom, so only 2 inputs
        p3_out = self.convs[3](
            self.bu_fusions[0]([p3_in, F.interpolate(p4_td, scale_factor=2, mode='nearest')])
        )
        
        # P4_out = Conv(Fusion(P4_in, P4_td, Pool(P3_out)))
        p4_out = self.convs[4](
            self.bu_fusions[1]([p4_in, p4_td, F.max_pool2d(p3_out, kernel_size=3, stride=2, padding=1)])
        )
        
        # P5_out = Conv(Fusion(P5_in, P5_td, Pool(P4_out)))
        p5_out = self.convs[5](
            self.bu_fusions[2]([p5_in, p5_td, F.max_pool2d(p4_out, kernel_size=3, stride=2, padding=1)])
        )
        
        # P6_out = Conv(Fusion(P6_in, P6_td, Pool(P5_out)))
        p6_out = self.convs[6](
            self.bu_fusions[3]([p6_in, p6_td, F.max_pool2d(p5_out, kernel_size=3, stride=2, padding=1)])
        )
        
        # P7_out = Conv(Fusion(P7_in, Pool(P6_out))) -> Note: P7_in, not P7_td
        p7_out = self.convs[7](
            self.bu_fusions[4]([p7_in, F.max_pool2d(p6_out, kernel_size=3, stride=2, padding=1)])
        )
        
        return [p3_out, p4_out, p5_out, p6_out, p7_out]

In [4]:
class BiFPNNetwork(nn.Module):
    def __init__(self, fpn_channels=64, num_layers=3):
        super().__init__()
        # Example 1x1 convs to project backbone features to fpn_channels
        # Assuming backbone outputs C3=512, C4=1024, C5=2048 channels
        self.p3_in = nn.Conv2d(512, fpn_channels, 1)
        self.p4_in = nn.Conv2d(1024, fpn_channels, 1)
        self.p5_in = nn.Conv2d(2048, fpn_channels, 1)
        
        # Create P6 and P7 from P5
        self.p6_in = nn.Conv2d(2048, fpn_channels, 3, stride=2, padding=1)
        self.p7_in = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(fpn_channels, fpn_channels, 3, stride=2, padding=1)
        )
        
        # Stack BiFPN blocks
        self.bifpn_layers = nn.Sequential(*[
            BiFPNBlock(fpn_channels) for _ in range(num_layers)
        ])

    def forward(self, c3, c4, c5):
        # 1. Project backbone features
        p3 = self.p3_in(c3)
        p4 = self.p4_in(c4)
        p5 = self.p5_in(c5)
        
        # 2. Generate P6 and P7
        p6 = self.p6_in(c5)
        p7 = self.p7_in(p6)
        
        features = [p3, p4, p5, p6, p7]
        
        # 3. Pass through BiFPN blocks
        features = self.bifpn_layers(features)
        
        return features

In [5]:
m2 = BiFPNNetwork()

In [6]:
m2

BiFPNNetwork(
  (p3_in): Conv2d(512, 64, kernel_size=(1, 1), stride=(1, 1))
  (p4_in): Conv2d(1024, 64, kernel_size=(1, 1), stride=(1, 1))
  (p5_in): Conv2d(2048, 64, kernel_size=(1, 1), stride=(1, 1))
  (p6_in): Conv2d(2048, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (p7_in): Sequential(
    (0): ReLU()
    (1): Conv2d(64, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  )
  (bifpn_layers): Sequential(
    (0): BiFPNBlock(
      (convs): ModuleList(
        (0-7): 8 x Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
          (1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
          (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (3): SiLU(inplace=True)
        )
      )
      (td_fusions): ModuleList(
        (0-2): 3 x WeightedFeatureFusion()
      )
      (bu_fusions): ModuleList(
        (0-4): 5 x WeightedFeatureFusion()
      )
    )
    

In [18]:
c3 = torch.randn(256,224,224)
c4 = torch.randn(512,224,224)
c5 = torch.randn(1024,224,224)

In [20]:
class YOLO_BiFPN_Adapter(nn.Module):
    def __init__(self, backbone_channels, fpn_channels=160, num_bifpn_layers=3):
        """
        Args:
            backbone_channels: List of ints, e.g., [256, 512, 1024] for P3, P4, P5
            fpn_channels: The fixed channel size for BiFPN (W_bifpn)
            num_bifpn_layers: How many BiFPN blocks to stack
        """
        super().__init__()
        
        c3, c4, c5 = backbone_channels
        
        # 1. Lateral Connections (1x1 Convs to match channel sizes)
        self.p3_proj = nn.Conv2d(c3, fpn_channels, kernel_size=1, stride=1, bias=False)
        self.p3_bn = nn.BatchNorm2d(fpn_channels)
        
        self.p4_proj = nn.Conv2d(c4, fpn_channels, kernel_size=1, stride=1, bias=False)
        self.p4_bn = nn.BatchNorm2d(fpn_channels)
        
        self.p5_proj = nn.Conv2d(c5, fpn_channels, kernel_size=1, stride=1, bias=False)
        self.p5_bn = nn.BatchNorm2d(fpn_channels)
        
        # 2. P6 Generation: Standard Conv 3x3 stride 2 from P5
        self.p6_gen = nn.Sequential(
            nn.Conv2d(c5, fpn_channels, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(fpn_channels)
        )
        
        # 3. P7 Generation: MaxPool from P6 (EfficientDet style) or Conv
        self.p7_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        # 4. The BiFPN Stack
        self.bifpn = nn.Sequential(*[
            BiFPNBlock(fpn_channels) for _ in range(num_bifpn_layers)
        ])
        
    def forward(self, features):
        # features is list [P3, P4, P5] from YOLO Backbone
        c3, c4, c5 = features
        
        # Project to FPN channels
        p3 = self.p3_bn(self.p3_proj(c3))
        p4 = self.p4_bn(self.p4_proj(c4))
        p5 = self.p5_bn(self.p5_proj(c5))
        
        # Generate extra levels
        p6 = self.p6_gen(c5) # Generate from original C5 containing rich info
        p7 = self.p7_pool(p6)
        
        # Feed into BiFPN stack
        # Input order matches BiFPNBlock: [P3, P4, P5, P6, P7]
        bifpn_out = self.bifpn([p3, p4, p5, p6, p7])
        
        return bifpn_out

In [21]:
class YoloBiFPNModel(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone # Your CSPDarknet/EfficientRep
        
        # Get backbone output channels, e.g., [256, 512, 1024]
        backbone_channels = self.backbone.output_channels 
        
        # The BiFPN Neck
        self.neck = YOLO_BiFPN_Adapter(backbone_channels, fpn_channels=160, num_bifpn_layers=3)
        
        # The Head (Assuming a standard YOLO Det Head)
        # Note: Input channels for the head are now [160, 160, 160]!
        self.head = YOLODetectHead(in_channels=[160, 160, 160], num_classes=num_classes)
        
    def forward(self, x):
        # Backbone
        backbone_feats = self.backbone(x) # [c3, c4, c5]
        
        # Neck
        # bifpn_feats has 5 items: [p3, p4, p5, p6, p7]
        bifpn_feats = self.neck(backbone_feats)
        
        # Head - We only take the first 3 for standard YOLO detection
        # (Or take all 5 if you customized your anchor generators)
        predictions = self.head([bifpn_feats[0], bifpn_feats[1], bifpn_feats[2]])
        
        return predictions

In [22]:
# Add this to ultralytics/nn/modules/block.py

class BiFPN_Concat(nn.Module):
    """
    BiFPN Weighted Fusion Layer.
    Fuses N inputs using learnable weights: O = sum(w_i * I_i) / (sum(w_i) + eps)
    """
    def __init__(self, dimension=1):
        super().__init__()
        self.d = dimension
        self.eps = 1e-4
        # We don't know N inputs at init (YOLO parser limitation), 
        # so we create a dynamic list or fix it if you prefer. 
        # Here we assume standard 2 or 3 inputs for BiFPN.
        # We'll initialize weights lazily or default to 3 (common max).
        self.w = nn.Parameter(torch.ones(3, dtype=torch.float32), requires_grad=True)

    def forward(self, x):
        # x is a list of tensors
        if not isinstance(x, list):
            return x
        
        n = len(x)
        # Dynamic slicing if fewer than 3 inputs
        w = self.w[:n]
        
        # Fast Normalized Fusion
        w_relu = F.relu(w)
        w_norm = w_relu / (w_relu.sum() + self.eps)
        
        # Weighted sum
        res = 0
        for i, tensor in enumerate(x):
            res += w_norm[i] * tensor
            
        return res